In [ ]:
from fastapi import APIRouter, HTTPException, status, Depends

from app.schemas.space import (
    SpaceCreate,
    SpaceUpdate,
    SpaceResponse,
)
from app.api.auth import get_current_user
from app.utils.helpers import generate_id, utc_now


router = APIRouter(
    prefix="/spaces",
    tags=["Spaces"],
)


def get_database():
    """Return the application's MongoDB wrapper."""
    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


def space_to_response(space: dict) -> SpaceResponse:
    """Convert a MongoDB document into the API response model."""

    return SpaceResponse(
        id=space["id"],
        user_id=space["user_id"],
        name=space["name"],
        description=space.get("description"),
        created_at=space["created_at"],
        updated_at=space["updated_at"],
    )


@router.post(
    "",
    response_model=SpaceResponse,
    status_code=status.HTTP_200_OK,
)
async def create_space(
    request: SpaceCreate,
    current_user=Depends(get_current_user),
):
    """Create a learning Space for the authenticated user."""

    database = get_database()
    spaces = database.collection("spaces")

    now = utc_now()

    space = {
        "id": generate_id(),
        "user_id": current_user.id,
        "name": request.name.strip(),
        "description": (
            request.description.strip()
            if request.description
            else None
        ),
        "created_at": now,
        "updated_at": now,
    }

    spaces.insert_one(space)

    return space_to_response(space)


@router.get(
    "",
    response_model=list[SpaceResponse],
)
async def list_spaces(
    current_user=Depends(get_current_user),
):
    """List Spaces owned by the authenticated user."""

    database = get_database()
    spaces = database.collection("spaces")

    documents = spaces.find(
        {"user_id": current_user.id}
    ).sort(
        "created_at",
        -1,
    )

    return [
        space_to_response(space)
        for space in documents
    ]


@router.get(
    "/{space_id}",
    response_model=SpaceResponse,
)
async def get_space(
    space_id: str,
    current_user=Depends(get_current_user),
):
    """Return one authorized Space."""

    database = get_database()
    spaces = database.collection("spaces")

    space = spaces.find_one(
        {
            "id": space_id,
            "user_id": current_user.id,
        }
    )

    if space is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Space not found.",
        )

    return space_to_response(space)


@router.put(
    "/{space_id}",
    response_model=SpaceResponse,
)
async def update_space(
    space_id: str,
    request: SpaceUpdate,
    current_user=Depends(get_current_user),
):
    """Update an authorized Space."""

    database = get_database()
    spaces = database.collection("spaces")

    existing = spaces.find_one(
        {
            "id": space_id,
            "user_id": current_user.id,
        }
    )

    if existing is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Space not found.",
        )

    update_data = {}

    if request.name is not None:
        update_data["name"] = request.name.strip()

    if request.description is not None:
        update_data["description"] = request.description.strip()

    update_data["updated_at"] = utc_now()

    spaces.update_one(
        {
            "id": space_id,
            "user_id": current_user.id,
        },
        {
            "$set": update_data,
        },
    )

    updated = spaces.find_one(
        {
            "id": space_id,
            "user_id": current_user.id,
        }
    )

    return space_to_response(updated)


@router.delete(
    "/{space_id}",
    status_code=status.HTTP_204_NO_CONTENT,
)
async def delete_space(
    space_id: str,
    current_user=Depends(get_current_user),
):
    """Delete an authorized Space."""

    database = get_database()
    spaces = database.collection("spaces")

    result = spaces.delete_one(
        {
            "id": space_id,
            "user_id": current_user.id,
        }
    )

    if result.deleted_count == 0:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Space not found.",
        )

    return None